<a href="https://colab.research.google.com/github/Ragul2526/RAG-based-Document-QA-Assistant/blob/main/qa_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Load PDF

We use PyMuPDF ('fitz') to open a PDF and extract its text page by page. This gives us a list of Strings - one per page, that we will later chunk and embed.

Upload any pdf to test.

##Make sure your Colab runtime is set to GPU:
##Runtime -> Change runtime type -> T4 GPU

In [ ]:
!pip install pymupdf -q
import torch

import fitz
from google.colab import files
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device: {device}")
if device == "cpu":
    print("Warning: No GPU found. Generation will be very slow.")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

print("Upload a pdf file:")
f = files.upload()
f_name = list(f.keys())[0]

print("Loaded ", f_name)

doc = fitz.open(f_name)
pages = [page.get_text() for page in doc]

print(f"Total pages :{len(pages)}")
print("==== Preview of the content ====")
print(pages[0][:500])

##Chunk the text

Pages are too large and uneven to search over directly.
We split all page text into fixed size overlapping chunks.

- chunk_size = 500 chars -> small enough to be focused
- chunk_overlap = 100 chars -> prevent key sentences being cut off at bounderies

Each chunk stores the page it came from so it will be easier to reference later.

In [ ]:
def chunk_text(pages, chunk_size = 500, chunk_overlap = 100):
  chunks = []
  chunk_index = 0
  for page_num, page_text in enumerate(pages):
    page_text = page_text.strip()
    if not page_text:
      continue
    s = 0
    while s < len(page_text):
      e = s + chunk_size
      chunk = page_text[s:e]

      chunks.append(
          {
           "text" : chunk,
           "page_number" : page_num + 1,
           "chunk_index" : chunk_index
          }
      )
      chunk_index += 1
      s += chunk_size - chunk_overlap
  return chunks

chunks = chunk_text(pages)
print("Total chunks created : ",len(chunks), ", from ", len(pages), "pages")

for i in range(3):
  c = chunks[i]
  print("Chunk -",c["chunk_index"],",Page number:",c["page_number"])
  print(c["text"],"\n")
print("-- Overlap check (last 100 chars of chunk 0 vs first 100 of chunk 1) --")
print("END of chunk 0:", repr(chunks[0]['text'][-100:]))
print("START of chunk 1:", repr(chunks[1]['text'][:100]))

##Embed Each Chunks

We convert every chunk's text into a 384-dimensional vector using
the `all-MiniLM-L6-v2` sentence embedding model.

These vectors capture *semantic meaning*, not just keywords.
Later, when a user asks a question, we embed the question the same way
and find whichever chunk vectors are closest - that's retrieval.

In [ ]:
!pip install sentence_transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embed Model loaded!\n")

chunk_texts = [c["text"] for c in chunks]

print("Embedding ",len(chunk_texts),"chunks")
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)

print(f"Done Embedding\n Embedding shape = {embeddings.shape}")

from numpy.linalg import norm
def cosine_sim(a,b):
  return np.dot(a,b)/(norm(a) * norm(b))
sim_adj = cosine_sim(embeddings[0],embeddings[1])
sim_far = cosine_sim(embeddings[0],embeddings[-1])
print(f"\n-- Similarity check --")
print(f"Chunk 0 vs Chunk 1 (adjacent):  {sim_adj:.3f}")
print(f"Chunk 0 vs Chunk -1 (far away): {sim_far:.3f}")
print("\n(Adjacent chunks should score higher — they share overlapping text!)")

##Build a FAISS index

FAISS lets us search across all chunk embeddings in milliseconds.
We use IndexFlatL2 - the simplest index type which does exact nearest-neighbor
search using L2 (Euclidean) distance.

For our small PDF this is perfect. For millions of chunks you'd swap this
for IndexIVFFlat (approximate but much faster at scale).

In [ ]:
!pip install faiss-cpu -q
import faiss

embeddingsf32 = np.array(embeddings).astype("float32")

dim = embeddingsf32.shape[1]
print("Embeddings Dimension : ",dim)

index = faiss.IndexFlatL2(dim)

index.add(embeddingsf32)

print(f"Vectors stored in index : {index.ntotal}")

query_vec = embeddingsf32[0:1]

k = 3
dist , ind = index.search(query_vec, k)

print(f"Top {k} nearest chunk indices: {ind[0]}")
print(f"Their L2 distances:            {dist[0]}\n")
print("Chunk 0 should be its own nearest neighbor (distance ≈ 0.0)")

for rank, (idx, dist) in enumerate(zip(ind[0], dist[0])):
    preview = chunks[idx]['text'][:80].replace('\n', ' ')
    print(f"  Rank {rank+1} | chunk {idx} | dist={dist:.4f} | \"{preview}...\"")


##Retrival Function

Given a question, it returns the top-K most semantically relevant chunks.

This is the "R" in RAG - Retrieval-Augmented Generation.
The retrieved chunks become the context we pass to the LLM

In [ ]:
def retrieve(question, k = 3):
  """
  When giving a plain question the function returns the top k most relevent chunks.

  Input :
    question : the user's question
    k : how many chunks to return

  Output :
    list of dicts, each with keys: text, page_number, chunk_index, distance

  """
  query_vec = embed_model.encode([question]).astype("float32")

  dist, ind = index.search(query_vec, k)

  results = []
  for dist,idx in zip(dist[0],ind[0]):
    chunk = chunks[idx].copy()
    chunk["distance"] = round(float(dist),4)
    results.append(chunk)
  return results

test_questions = [
    "What are the main skills mentioned?",
    "What projects has this person worked on?",
    "What is their educational background?",
]
for q in test_questions:
  print(f"Question : {q}")
  results = retrieve(q)
  for rank, r in enumerate(results):
    p = r ["text"][:150].replace("\n"," ")
    print(f" Rank {rank + 1} | Page {r["page_number"]} | distance = {r["distance"]}\n {p} \n")


##Load the LLM

We load Qwen2.5-1.5B-Instruct from HuggingFace using the `transformers` library.
Every local LLM needs two things:
- Tokenizer: converts text ↔ token IDs
- Model: the actual neural network that generates text

We use float16 precision and move the model to GPU to keep memory usage low.


In [ ]:
#!pip install transformers accelerate -q
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"




tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Tokenizer loaded")

print("This downloads the model ~3GB, it takes time for the first time")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("Model loaded")

sample = "Hello I am a LM model"

t = tokenizer(sample, return_tensors="pt")
print("--Tokenizer check--")
print(f"Text : {sample}\nTokenized : {t['input_ids'][0].tolist()}\nDecoded back : {tokenizer.decode(t['input_ids'][0])} ")
if device == "cuda":
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU memory used: {used:.1f} GB / {total:.1f} GB total")

##RAG prompt + Generation Function

We build two functions:
- `build_rag_prompt()` - formats retrieved chunks + question into a prompt
- `gen_answer()` - runs the prompt through Qwen and returns the answer

The system message instructs the model to answer ONLY from context.
This is the key constraint that makes RAG reduce hallucination.

In [ ]:
def build_rag_prompt(question,  retrieved_chunks):
  """
  This function combines retrieved chunks + question into a chat-style prompt.

  Input:
    question - User's question
    retrieved_chunks - Output of retrieve()

  Output:
    list of message dicts in OpenAI/HuggingFace chat format

  """
  context_blocks = []
  for i,chunk in enumerate(retrieved_chunks):
    block = f"[Source {i+1} - Page {chunk['page_number']}]\n{chunk['text'].strip()}"
    context_blocks.append(block)

  context_str = "\n\n".join(context_blocks)
  #print(f"\nContext:{context_str}\n")
  messages = [
        {
            "role": "user",
            "content": (
                f"Here is context extracted from a document:\n\n"
                f"{context_str}\n\n"
                f"---\n"
                f"Using ONLY the context above, answer this question. "
                f"Do NOT use outside knowledge. "
                f"If the answer is not in the context, say 'I could not find that in the document.'\n\n"
                f"Question: {question}"
            )
        }
    ]


  return messages

def gen_answer(question, k = 3, max_new_tokens = 300):

  """
  This Function represents the full RAG pipeline: retrieve ->  prompt -> generate -> return answer to user

  Input:
    question - user's question
    k - how many chunks to retrieve
    max_new_tokens : max length of the answer

  Output:
    dict with keys: answer, retrieved_chunks

  """
  retrieve_chunks = retrieve(question, k)
  messages = build_rag_prompt(question, retrieve_chunks)

  tokenized = tokenizer.apply_chat_template( # apply_chat_template formats messages into the exact string Qwen expects
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)
  if isinstance(tokenized, dict) or hasattr(tokenized, 'input_ids'):
    input_ids = tokenized["input_ids"].to(device)
  else:
    input_ids = tokenized.to(device)
  with torch.no_grad():
    output_ids = model.generate(
          input_ids=input_ids,
          max_new_tokens=max_new_tokens,
          do_sample=False,        # greedy decoding → deterministic output
          temperature=None,       # must be None when do_sample=False
          top_p=None,             # must be None when do_sample=False
          pad_token_id=tokenizer.eos_token_id)
  new_tokens = output_ids[0][input_ids.shape[-1]:]
  answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
  return {"answer" : answer.strip(), "retrieved_chunks" : retrieve_chunks}


print("--Testing RAG pipeline--")
test_q = "What are the main skills mentioned in the document?"
result = gen_answer(test_q, k=3)

print("Question :",test_q)
print(f"\nAnswer:\n{result['answer']}")
print(f"\nRetrieved from:")

for chunk in result["retrieved_chunks"]:
  print(f" Page {chunk['page_number']} | dist={chunk['distance']} | {chunk['text'][:80].replace(chr(10), ' ')}")


## RAG vs No-RAG Comparison

We ask the same questions two ways:
- Without retrieval: LLM answers purely from its training data (may hallucinate)
- With retrieval: LLM answers only from the document context (grounded)

This comparison visually proves why RAG exists.

In [ ]:

def gen_without_rag(question, max_new_tokens=300):
    """
    This function asks the LLM the question directly — no context, no retrieval.
    The model must rely purely on its training data → likely to hallucinate
    on personal/specific documents it has never seen.
    """
    messages = [
        {
            "role": "user",
            "content": f"Answer this question as best you can:\n\n{question}"
        }
    ]

    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if isinstance(tokenized, dict) or hasattr(tokenized, 'input_ids'):
        input_ids = tokenized["input_ids"].to(device)
    else:
        input_ids = tokenized.to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()



comparison_questions = [
    "What is the candidate's name and where are they located?",
    "What machine learning frameworks does the candidate know?",
    "What is the candidate's educational qualification?",
]

for q in comparison_questions:
    print("=" * 70)
    print(f"QUESTION:\n   {q}")
    print("=" * 70)

    # Without RAG
    print("\nWITHOUT RAG (no context - LLM guesses):")
    print("─" * 50)
    no_rag_answer = gen_without_rag(q)
    print(no_rag_answer)

    # With RAG
    print("\nWITH RAG (grounded in your document):")
    print("─" * 50)
    rag_result = gen_answer(q, k=3)
    print(rag_result['answer'])
    print(f"\nRetrieved from: {[f'Page {c["page_number"]}' for c in rag_result['retrieved_chunks']]}\n")

##RAG vs No-RAG — What We Learned

We tested the same questions with and without retrieval.

WITHOUT RAG → Qwen correctly admits it doesn't know (it has never seen your resume)
WITH RAG    → Qwen answers precisely using retrieved chunks

This reveals the REAL value of RAG:

Without RAG = "I cannot answer without more context" (useless for private docs)
With RAG    = "RagulRajkumar S, Chennai. Skills: PyTorch, TensorFlow..." (precise)

RAG doesn't just fix hallucination.
It makes private document Q&A possible in the first place.
A model that has never seen your resume becomes an expert on it — instantly.
That's the point.

##Gradio App

We wrap the entire RAG pipeline into a Gradio interface.
- Upload any PDF -> chunks + embeddings + FAISS index rebuild automatically
- Ask any question -> retrieval + generation runs instantly
- Sources shown below every answer -> fully auditable

I am going to prepare a separate app.py and push it to github and huggingface(Using Gradio) to deploy the Q&A Assistant.
